# Log Analysis — Processing Durations

Parses logs, extracts per-frame processing durations, builds a summary DataFrame, plots distributions, and highlights outliers.

In [ ]:
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd
from matplotlib import pyplot as plt

## Extract data

In [ ]:
# parse file into dict
pattern = re.compile(r"DEBUG processing: \(([0-9a-f]{6})\) (.+?) duration: ([\d.]+) ms")

raw: dict[str, dict] = defaultdict(dict)

with Path("../gdrive/20260408_124640_logs.txt").open() as f:
    for line in f:
        m = pattern.search(line)
        if not m:
            continue
        frame_hash, step, value = m.group(1), m.group(2), float(m.group(3))
        col = step.lower().replace(" ", "_") + "_ms"
        raw[frame_hash][col] = value

In [ ]:
# convert to dataframe
df = pd.DataFrame.from_dict(raw, orient="index")
df.index.name = "hash"
df_with_obj = df.loc[df["object_detection_ms"].notna()]
normalised = df.reset_index(drop=True) / df_with_obj.median()
df.describe()

## Duration evolution

In [ ]:
fig, ax = plt.subplots()

normalised.plot(logy=True, ax=ax)
ax.set_ylabel("× median")
ax.set_xlabel("Frame index")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()

## Distribution of Processing Durations

In [ ]:
fig, axes = plt.subplots(1, df.shape[1], figsize=(18, 4))
fig.suptitle("Distribution of Processing Durations (ms)")

for ax, (col) in zip(axes, df.columns):
    data = df[col].dropna()
    ax.hist(data, bins=30)
    ax.set_title(col)
    ax.set_xlabel("Duration (ms)")
    ax.set_ylabel("Count")

fig.tight_layout()

## Outlier Detection

In [ ]:
def iqr_upper_bound(series: pd.Series) -> float:
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    return q3 + 1.5 * (q3 - q1)


records = []
for col in df.columns:
    if col == "processing_ms":
        data = df_with_obj[col].dropna()
    else:
        data = df[col].dropna()
    hi = iqr_upper_bound(data)
    outliers = data[data > hi]
    for hash_val, val in outliers.items():
        records.append(
            {
                "hash": hash_val,
                "column": col,
                "value_ms": val,
                "upper_bound": hi,
                "multiple": val / hi,
            }
        )

df_out = pd.DataFrame(records)
df_out.sort_values("multiple", ascending=False)

## Frame rate analysis

In [ ]:
# quantile frame rate possibilities

print(f"Processing duration for {len(df_with_obj)} frames with object detection: (ms):")
print(
    f"  p80    {df_with_obj["processing_ms"].quantile(0.80):.1f}  →  {1000/df_with_obj["processing_ms"].quantile(0.80):.1f} fps"
)
print(
    f"  p90    {df_with_obj["processing_ms"].quantile(0.90):.1f}  →  {1000/df_with_obj["processing_ms"].quantile(0.90):.1f} fps"
)
print(
    f"  p95    {df_with_obj["processing_ms"].quantile(0.95):.1f}  →  {1000/df_with_obj["processing_ms"].quantile(0.95):.1f} fps"
)
print(
    f"  p99    {df_with_obj["processing_ms"].quantile(0.99):.1f}  →  {1000/df_with_obj["processing_ms"].quantile(0.99):.1f} fps"
)
print(
    f"  max    {df_with_obj["processing_ms"].max():.1f}  →  {1000/df_with_obj["processing_ms"].max():.1f} fps"
)

In [ ]:
# breakdown of frame driving processes
means = df_with_obj.mean()
medians = df_with_obj.median()

summary = pd.DataFrame(
    {
        "mean_ms": means.drop(columns="processing_ms"),
        "median_ms": medians.drop(columns="processing_ms"),
    }
)
summary.round(2)